In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install rank_bm25 faiss-cpu sentence-transformers -q

import torch
import torch.nn as nn
import json
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import faiss

DRIVE_BASE     = "/content/drive/MyDrive/medrag"
DATA_PATH      = f"{DRIVE_BASE}/pubmedqa_filtered.json"
CHECKPOINT_DIR = f"{DRIVE_BASE}/biomistral_lens_checkpoints"
OUTPUT_DIR     = f"{DRIVE_BASE}/biomistral_trial4_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Corpus: {len(corpus)} samples")

In [ ]:
MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    use_safetensors=False
)
model = model.to("cuda")
model.eval()
for param in model.parameters():
    param.requires_grad = False

n_layers    = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vocab_size  = model.config.vocab_size

print(f"Model: {n_layers} layers, hidden {hidden_size}, vocab {vocab_size}")

In [ ]:
class TunedLensTranslator(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.translator = nn.Linear(hidden_size, hidden_size, bias=True)
        nn.init.eye_(self.translator.weight)
        nn.init.zeros_(self.translator.bias)

    def forward(self, hidden_state):
        return self.translator(hidden_state.float())

translators = nn.ModuleList([TunedLensTranslator(hidden_size) for _ in range(n_layers)])

for layer_idx in range(n_layers):
    ckpt = torch.load(
        os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt"),
        map_location="cuda"
    )
    translators[layer_idx].load_state_dict(ckpt["state_dict"])

translators = translators.to("cuda")
translators.eval()
for param in translators.parameters():
    param.requires_grad = False

RAW_LENS_LAYERS = set(range(17, 31))

print(f"Tuned-lens loaded: {n_layers} translators")
print(f"Raw logit lens layers: {sorted(RAW_LENS_LAYERS)}")

In [ ]:
def project_hidden(hidden_state, layer_idx):
    with torch.no_grad():
        h = hidden_state if hidden_state.dim() == 2 else hidden_state.unsqueeze(0)

        if layer_idx in RAW_LENS_LAYERS:
            normed = model.model.norm(h.half())
            logits = model.lm_head(normed).float()
        else:
            translated = translators[layer_idx](h.float())
            normed     = model.model.norm(translated.half())
            logits     = model.lm_head(normed).float()

        return torch.softmax(logits.squeeze(0), dim=-1)

print("Projection function ready.")

In [ ]:
documents, doc_metadata = [], []
for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({"pubid": sample["pubid"], "label": sample["label"]})

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(
    documents, batch_size=32, show_progress_bar=True, convert_to_numpy=True
)
faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready: {index.ntotal} documents indexed")

In [ ]:
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k  = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "score": scores[i]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "score": float(s)}
            for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    combined = {}
    for rank, r in enumerate(bm25_retrieve(query, k=k*2)):
        combined[r["abstract"]] = {"meta": r, "score": 1/(rrf_k+rank+1)}
    for rank, r in enumerate(faiss_retrieve(query, k=k*2)):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1/(rrf_k+rank+1)
        else:
            combined[key] = {"meta": r, "score": 1/(rrf_k+rank+1)}
    return [v["meta"] for v in sorted(
        combined.values(), key=lambda x: x["score"], reverse=True)[:k]]

def build_rag_prompt(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context = "\n\n".join(f"[{i+1}] {r['abstract']}" for i, r in enumerate(results))
    return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

def build_suppressed_prompt(query):
    return f"Question: {query}\nAnswer:"

print("Prompt functions ready.")

In [ ]:
hook_storage = {}
hooks = []

def make_hook(li):
    def hook(module, inp, output):
        hook_storage[li] = output[0].detach().squeeze(0)
    return hook

for i, block in enumerate(model.model.layers):
    hooks.append(block.register_forward_hook(make_hook(i)))

N_SAMPLES      = len(corpus)
trial4_results = []

print(f"Running Trial 4 (KL RAG vs suppressed) on {N_SAMPLES} samples...\n")

for idx, sample in enumerate(tqdm(corpus[:N_SAMPLES])):
    query = sample["query"]

    rag_prompt = build_rag_prompt(query)
    rag_inputs = tokenizer(
        rag_prompt, return_tensors="pt",
        truncation=True, max_length=512
    ).to("cuda")

    hook_storage.clear()
    with torch.no_grad():
        model(**rag_inputs)
    rag_hidden = {k: v.clone() for k, v in hook_storage.items()}

    sup_prompt = build_suppressed_prompt(query)
    sup_inputs = tokenizer(
        sup_prompt, return_tensors="pt",
        truncation=True, max_length=512
    ).to("cuda")

    hook_storage.clear()
    with torch.no_grad():
        model(**sup_inputs)
    sup_hidden = {k: v.clone() for k, v in hook_storage.items()}

    kl_trajectory = []

    for layer_idx in range(n_layers):
        p_rag = project_hidden(rag_hidden[layer_idx][-1, :], layer_idx)
        p_sup = project_hidden(sup_hidden[layer_idx][-1, :], layer_idx)

        kl = float(torch.sum(
            p_rag * (torch.log(p_rag + 1e-10) - torch.log(p_sup + 1e-10))
        ).item())
        kl_trajectory.append(kl)

    trial4_results.append({
        "idx":            idx,
        "pubid":          sample["pubid"],
        "query":          query[:60],
        "label":          sample["label"],
        "rag_seq_len":    rag_inputs["input_ids"].shape[1],
        "sup_seq_len":    sup_inputs["input_ids"].shape[1],
        "kl_trajectory":  kl_trajectory,
        "mean_kl":        float(np.mean(kl_trajectory[1:])),
        "max_kl_layer":   int(np.argmax(kl_trajectory[1:]) + 1),
        "max_kl_value":   float(np.max(kl_trajectory[1:])),
        "final_layer_kl": kl_trajectory[-1],
    })

    if (idx + 1) % 50 == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f"trial4_checkpoint_{idx+1}.json")
        with open(ckpt_path, "w") as f:
            json.dump(trial4_results, f)
        torch.cuda.empty_cache()
        tqdm.write(f"Checkpoint saved at sample {idx+1}")

for h in hooks:
    h.remove()

print(f"\nTrial 4 complete. {len(trial4_results)} samples processed.")

In [ ]:
print("TRIAL 4 SUMMARY BY LABEL\n")
for label in ["yes", "no"]:
    ls = [r for r in trial4_results if r["label"] == label]
    print(f"Label: {label} | n={len(ls)}")
    print(f"  Mean KL (RAG||sup):   {np.mean([r['mean_kl'] for r in ls]):.4f}")
    print(f"  Mean peak layer:      {np.mean([r['max_kl_layer'] for r in ls]):.1f}")
    print(f"  Mean final layer KL:  {np.mean([r['final_layer_kl'] for r in ls]):.4f}")
    print()

print("BioMedLM T4 reference:")
print("  yes=0.254, no=0.298 (17% difference) | Peak layer: 23.8\n")

all_peak_layers = [r["max_kl_layer"] for r in trial4_results]
mean_peak = np.mean(all_peak_layers)
print(f"Overall mean peak layer (use for activation patching): {mean_peak:.1f}")
print(f"Suggested PATCH_LAYER = {int(round(mean_peak))}\n")

output_path = os.path.join(OUTPUT_DIR, "trial4_full_results.json")
with open(output_path, "w") as f:
    json.dump({
        "model_id":        MODEL_ID,
        "n_samples":       len(trial4_results),
        "n_layers":        n_layers,
        "raw_lens_layers": sorted(RAW_LENS_LAYERS),
        "summary_by_label": {
            label: {
                "n":               sum(1 for r in trial4_results if r["label"] == label),
                "mean_kl":         float(np.mean([r["mean_kl"]         for r in trial4_results if r["label"] == label])),
                "mean_peak_layer": float(np.mean([r["max_kl_layer"]    for r in trial4_results if r["label"] == label])),
                "mean_final_kl":   float(np.mean([r["final_layer_kl"]  for r in trial4_results if r["label"] == label])),
            } for label in ["yes", "no"]
        },
        "samples": trial4_results
    }, f, indent=2)

for fname in os.listdir(OUTPUT_DIR):
    if fname.startswith("trial4_checkpoint"):
        os.remove(os.path.join(OUTPUT_DIR, fname))

print(f"Results saved to {output_path}")
print("09_biomistral_trial4_kl_rag_suppressed: complete")